# CNN Visualization & Adversarial Attacks

[← Back to lesson](https://ml-viz-ruby.vercel.app/courses/cnns/03-cnn-visualization-and-attacks)

This notebook simulates key CNN interpretability and robustness techniques: saliency maps, a toy Grad-CAM implementation, FGSM adversarial attack, and adversarial training. All examples use a simple 2-layer linear network on synthetic data so you can see the mechanics without needing a GPU.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.edgecolor': '#2d3748',
    'grid.color': '#2d3748',
    'axes.grid': True,
})

np.random.seed(0)

## Intuition — seeing what a model sees, and fooling it

Two sides of the same coin: the **gradient of the output with respect to the input**. Point it one
way and you get a **saliency map** — which pixels most affect the prediction, i.e. where the model
"looks." Point it the other way and you get an **adversarial attack** — nudge the input along the
gradient's sign (**FGSM**) and a small perturbation flips the prediction (on real
high-dimensional images, imperceptibly small; on this easy toy dataset it must be larger). The same
`∂output/∂input` that explains a model also breaks it. **Adversarial training** (training on attacked
examples) partially restores robustness. We build all three from scratch and gradient-check the
attack — exactly how you'd verify a real one.

## 1. Saliency maps — gradient of output w.r.t. input

We build a 2-class toy "image classifier" on 8×8 synthetic images (one class has a bright top-left region; the other has a bright bottom-right region). We then compute saliency maps to see if the network learned to look in the right places.

In [ ]:
IMG_SIZE = 8
FLAT = IMG_SIZE * IMG_SIZE

def make_dataset(n_per_class=200, noise=0.3, seed=0):
    rng = np.random.RandomState(seed)
    X, y = [], []
    for label in [0, 1]:
        for _ in range(n_per_class):
            img = rng.randn(IMG_SIZE, IMG_SIZE) * noise
            if label == 0:
                img[:4, :4] += 1.5  # top-left bright patch
            else:
                img[4:, 4:] += 1.5  # bottom-right bright patch
            X.append(img.ravel())
            y.append(label)
    return np.array(X), np.array(y)

X_train, y_train = make_dataset(200)
X_test, y_test = make_dataset(50, seed=1)

# Simple linear model: score = W @ x + b
class LinearClassifier:
    def __init__(self, n_features, n_classes):
        self.W = np.random.randn(n_classes, n_features) * 0.01
        self.b = np.zeros(n_classes)

    def forward(self, x):
        return x @ self.W.T + self.b  # (n, C)

    def softmax(self, logits):
        logits = logits - logits.max(axis=1, keepdims=True)
        exp = np.exp(logits)
        return exp / exp.sum(axis=1, keepdims=True)

    def loss(self, x, y):
        probs = self.softmax(self.forward(x))
        return -np.log(probs[np.arange(len(y)), y] + 1e-9).mean()

    def fit(self, X, y, lr=0.5, n_iters=500):
        n = len(y)
        for _ in range(n_iters):
            logits = self.forward(X)
            probs = self.softmax(logits)
            probs[np.arange(n), y] -= 1
            self.W -= lr * probs.T @ X / n
            self.b -= lr * probs.mean(axis=0)

    def predict(self, X):
        return self.forward(X).argmax(axis=1)

model = LinearClassifier(FLAT, 2)
model.fit(X_train, y_train)
acc = (model.predict(X_test) == y_test).mean()
print(f"Test accuracy: {acc:.1%}")

# --- Saliency map: gradient of class score w.r.t. input ---
# For a linear model: ∂score_c/∂x = W[c,:]
saliency_0 = np.abs(model.W[0]).reshape(IMG_SIZE, IMG_SIZE)
saliency_1 = np.abs(model.W[1]).reshape(IMG_SIZE, IMG_SIZE)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

# Example images from each class
x0 = X_test[y_test == 0][0].reshape(IMG_SIZE, IMG_SIZE)
x1 = X_test[y_test == 1][0].reshape(IMG_SIZE, IMG_SIZE)

for ax, img, title in zip(axes[:2], [x0, x1], ['Class 0 image\n(top-left)', 'Class 1 image\n(bottom-right)']):
    im = ax.imshow(img, cmap='viridis', vmin=-1, vmax=3)
    ax.set_title(title, color='#e2e8f0')
    ax.axis('off')

for ax, sal, title in zip(axes[2:], [saliency_0, saliency_1], ['Saliency (class 0)', 'Saliency (class 1)']):
    ax.imshow(sal, cmap='hot')
    ax.set_title(title, color='#e2e8f0')
    ax.axis('off')

plt.suptitle('Saliency Maps: Where Does the Model Look?', color='#e2e8f0')
plt.tight_layout()
plt.show()

print("Top-left quadrant saliency (class 0):", saliency_0[:4, :4].mean().round(4))
print("Bottom-right quadrant saliency (class 0):", saliency_0[4:, 4:].mean().round(4))

**What to notice:** for this linear model the saliency is just `|W|`, and it concentrates exactly
where each class's evidence lives — class-0 saliency in the **top-left** quadrant (its bright patch),
class-1 in the **bottom-right**. The model learned to look at the right region. (For deep nets the
saliency is the input gradient computed by backprop — same idea, more layers.)

## The library way — gradient-check the attack

An adversarial attack is only as correct as its input gradient. Before trusting FGSM's analytic
gradient `(p − onehot) @ W`, verify it against a **finite-difference** estimate of `∂loss/∂x` — the
same gradient check you'd run on any attack code. (In practice you'd use `torch` autograd with a
library like `foolbox`/`torchattacks`.)

In [ ]:
x = X_test[0].copy(); yt = int(y_test[0])

# analytic input gradient (what FGSM uses)
probs = model.softmax(model.forward(x[None]))[0].copy()
probs[yt] -= 1.0
grad_analytic = probs @ model.W

# finite-difference gradient of the loss w.r.t. the input pixels
eps = 1e-5
grad_num = np.zeros_like(x)
for i in range(len(x)):
    xp = x.copy(); xp[i] += eps
    xm = x.copy(); xm[i] -= eps
    grad_num[i] = (model.loss(xp[None], [yt]) - model.loss(xm[None], [yt])) / (2 * eps)

print('max |analytic - numerical| =', np.max(np.abs(grad_analytic - grad_num)))
assert np.allclose(grad_analytic, grad_num, atol=1e-4), "FGSM gradient must pass the finite-diff check"
print('FGSM input gradient verified against finite differences ✓')

**What to notice:** the analytic input gradient matches finite differences — so FGSM is stepping in
the genuinely loss-*increasing* direction. This gradient check is non-negotiable for attack code: a
sign error would produce a "defense" that actually helps the model, and you'd never know.

## 2. FGSM — Fast Gradient Sign Method

FGSM perturbs each input pixel by $\epsilon \cdot \text{sign}(\nabla_x J)$ to maximize the loss.

In [ ]:
def fgsm_attack(model, x, y_true, epsilon):
    """
    x: (FLAT,) single image
    Returns adversarial example.
    For a linear model: ∂loss/∂x = W[y_wrong,:] - W[y_true,:] (approximately)
    More precisely: gradient = (p - one_hot(y)) @ W
    """
    x_batch = x[np.newaxis, :]  # (1, FLAT)
    logits = model.forward(x_batch)
    probs = model.softmax(logits)[0]
    probs[y_true] -= 1.0          # dL/d(logits) = probs - one_hot
    grad_x = probs @ model.W      # (FLAT,)
    return x + epsilon * np.sign(grad_x)

# Attack accuracy across epsilon values
epsilons = np.linspace(0, 1.0, 20)
clean_accs, adv_accs = [], []

for eps in epsilons:
    X_adv = np.array([fgsm_attack(model, X_test[i], y_test[i], eps) for i in range(len(X_test))])
    clean_accs.append((model.predict(X_test) == y_test).mean())
    adv_accs.append((model.predict(X_adv) == y_test).mean())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epsilons, clean_accs, 'o-', color='#10b981', linewidth=2, label='Clean accuracy')
ax1.plot(epsilons, adv_accs, 's-', color='#f43f5e', linewidth=2, label='Adversarial accuracy')
ax1.set_xlabel('Perturbation ε')
ax1.set_ylabel('Accuracy')
ax1.set_title('FGSM Attack: Accuracy vs ε', color='#e2e8f0')
ax1.legend()
ax1.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))

# Visualize a clean vs adversarial example
idx = 0
eps_demo = 0.4
x_adv_demo = fgsm_attack(model, X_test[idx], y_test[idx], eps_demo)
perturbation = x_adv_demo - X_test[idx]

for i, (img, title) in enumerate([
    (X_test[idx].reshape(IMG_SIZE, IMG_SIZE), 'Original'),
    (perturbation.reshape(IMG_SIZE, IMG_SIZE), f'Perturbation (ε={eps_demo})'),
    (x_adv_demo.reshape(IMG_SIZE, IMG_SIZE), 'Adversarial'),
]):
    ax2.set_visible(False)  # placeholder

plt.tight_layout()
plt.show()

print(f"Clean prediction: {model.predict(X_test[idx:idx+1])[0]} (true: {y_test[idx]})")
print(f"Adversarial prediction (ε={eps_demo}): {model.predict(x_adv_demo[np.newaxis,:])[0]}")

**What to notice:** clean accuracy stays at 100%, and adversarial accuracy **holds until `ε≈0.8`,
then collapses toward 0**. Once the perturbation is large enough to overwhelm the strong class signal
(a `+1.5` patch), a single gradient step breaks *every* prediction. On harder, high-dimensional
images the threshold is far smaller — the famous "imperceptible" adversarial examples — but the
mechanism is identical: step along `sign(∂loss/∂x)`.

## 3. Adversarial training

Train on a mix of clean and FGSM-perturbed examples. Does this improve adversarial robustness?

In [ ]:
def adversarial_train(X_train, y_train, epsilon=0.3, n_iters=500, lr=0.5):
    model_adv = LinearClassifier(FLAT, 2)
    n = len(y_train)
    for it in range(n_iters):
        # Generate adversarial examples on the fly
        X_adv_batch = np.array([
            fgsm_attack(model_adv, X_train[i], y_train[i], epsilon)
            for i in range(n)
        ])
        # Train on mixed batch
        X_mix = np.vstack([X_train, X_adv_batch])
        y_mix = np.concatenate([y_train, y_train])
        logits = model_adv.forward(X_mix)
        probs = model_adv.softmax(logits)
        probs[np.arange(2*n), y_mix] -= 1
        model_adv.W -= lr * probs.T @ X_mix / (2*n)
        model_adv.b -= lr * probs.mean(axis=0)
    return model_adv

eps_train = 0.4
model_robust = adversarial_train(X_train, y_train, epsilon=eps_train)

test_eps = np.linspace(0, 1.0, 20)
adv_accs_standard, adv_accs_robust = [], []

for eps in test_eps:
    X_adv = np.array([fgsm_attack(model, X_test[i], y_test[i], eps) for i in range(len(X_test))])
    X_adv_r = np.array([fgsm_attack(model_robust, X_test[i], y_test[i], eps) for i in range(len(X_test))])
    adv_accs_standard.append((model.predict(X_adv) == y_test).mean())
    adv_accs_robust.append((model_robust.predict(X_adv_r) == y_test).mean())

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(test_eps, adv_accs_standard, 'o-', color='#f43f5e', linewidth=2, label='Standard training')
ax.plot(test_eps, adv_accs_robust, 's-', color='#10b981', linewidth=2, label=f'Adversarial training (ε={eps_train})')
ax.set_xlabel('Attack ε')
ax.set_ylabel('Adversarial accuracy')
ax.set_title('Adversarial Training Improves Robustness', color='#e2e8f0')
ax.legend()
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
plt.tight_layout()
plt.show()

clean_std = (model.predict(X_test) == y_test).mean()
clean_rob = (model_robust.predict(X_test) == y_test).mean()
print(f"Clean accuracy — standard: {clean_std:.1%}, robust: {clean_rob:.1%}")
print("Note the accuracy-robustness trade-off: robust model may have lower clean accuracy.")

**What to notice:** after **adversarial training** (training on FGSM-perturbed examples), the
model's accuracy under attack recovers substantially — it has learned to resist the perturbation
direction. This robustness usually costs a little clean accuracy, the classic robustness–accuracy
tradeoff.

## Gotchas & tradeoffs

- **Attacks are cheap, defenses are hard.** FGSM is one gradient step; robustly defending against all
  attacks remains unsolved. Adversarial training helps but isn't a cure.
- **Robustness costs clean accuracy.** Robust models typically give up some standard accuracy — you
  trade one for the other.
- **White-box vs black-box.** FGSM needs the model's gradients (white-box); real attackers often only
  have query access, yet attacks still **transfer** between models.
- **Saliency can mislead.** For deep nets raw-gradient saliency is noisy and can be manipulated; use
  smoothed/attribution methods (Grad-CAM, integrated gradients) and interpret with care.

In [ ]:
# A large-enough single-step perturbation flips predictions across the test set
eps = 1.0
X_adv = np.array([fgsm_attack(model, X_test[i], int(y_test[i]), eps) for i in range(len(X_test))])
flip_rate = np.mean(model.predict(X_adv) != y_test)
print(f'at eps={eps}: {flip_rate:.0%} of the (100%-correct) test images are now misclassified')
j = int(np.where(model.predict(X_adv) != y_test)[0][0])
print(f'example {j}: clean pred {model.predict(X_test[j:j+1])[0]} -> adversarial pred '
      f'{model.predict(X_adv[j:j+1])[0]} (true {int(y_test[j])})')

**What to notice:** at `ε=1.0` the one-step FGSM perturbation flips essentially the entire test set
from 100% correct to ~0%. On this strong-signal toy data the perturbation is *visible*; the alarming
real-world result is that for natural images in thousands of dimensions, a perturbation **too small
for a human to see** achieves the same — which is why adversarial robustness is a serious deployment
concern.

## Key takeaways

- **Saliency** = gradient of the output w.r.t. the input (where the model looks); for a linear model
  it's just `|W|`.
- **FGSM** steps the input along `sign(∂loss/∂x)` — a tiny perturbation that flips predictions. Always
  **gradient-check** attack code.
- **High accuracy ≠ robust.** Adversarial accuracy can collapse under imperceptible perturbations.
- **Adversarial training** restores some robustness at a small clean-accuracy cost; attacks
  **transfer**, so black-box defense is hard.

**Next:** [Transfer Learning](https://ml-viz-ruby.vercel.app/courses/cnns/04-transfer-learning).

## ✏️ Your turn

**Exercise 1 — PGD attack.** FGSM is a one-step attack. Implement PGD (Projected Gradient Descent) — iterate FGSM $k$ times with a smaller step size $\alpha = \epsilon/k$, clipping the cumulative perturbation to the ε-ball after each step. Compare PGD accuracy vs FGSM accuracy against both models.

In [ ]:
def pgd_attack(model, x, y_true, epsilon, alpha, k):
    """
    PGD attack: k steps of FGSM with step size alpha, clipped to ε-ball.
    TODO(you): implement this
    """
    x_adv = x.copy()
    # hint: for each step, apply FGSM with step alpha, then clip |x_adv - x| <= epsilon
    pass
    return x_adv

In [ ]:
# Assert cell
def pgd_ref(model, x, y_true, epsilon, alpha, k):
    x_adv = x.copy()
    for _ in range(k):
        x_adv = fgsm_attack(model, x_adv, y_true, alpha)
        x_adv = np.clip(x_adv, x - epsilon, x + epsilon)
    return x_adv

eps, alpha, k = 0.5, 0.1, 10
X_pgd = np.array([pgd_ref(model, X_test[i], y_test[i], eps, alpha, k) for i in range(len(X_test))])
X_fgsm = np.array([fgsm_attack(model, X_test[i], y_test[i], eps) for i in range(len(X_test))])
pgd_acc = (model.predict(X_pgd) == y_test).mean()
fgsm_acc = (model.predict(X_fgsm) == y_test).mean()
print(f"FGSM accuracy (ε={eps}): {fgsm_acc:.1%}")
print(f"PGD accuracy  (ε={eps}, k={k}): {pgd_acc:.1%}")
assert pgd_acc <= fgsm_acc + 0.05, "PGD should be at least as strong as FGSM"

<details><summary>Solution</summary>

```python
def pgd_attack(model, x, y_true, epsilon, alpha, k):
    x_adv = x.copy()
    for _ in range(k):
        x_adv = fgsm_attack(model, x_adv, y_true, alpha)  # one FGSM step
        x_adv = np.clip(x_adv, x - epsilon, x + epsilon)  # project back to ε-ball
    return x_adv
```

PGD is stronger than FGSM because it refines the perturbation over multiple steps. With $k = 10$ steps, it finds a more adversarial point within the ε-ball than the single FGSM step. This is why PGD is considered the standard attack for evaluating robustness — a model robust against PGD is robust against most known attacks.

</details>